# Residual Mechanization Index

This notebook develops a more complex measure of agricultural mechanization designed to better account for structural differences between Romanian counties, particularly differences in cultivated agricultural area.

The first, density-based mechanization index measures machinery intensity relative to cultivated area. In this second specification, mechanization is estimated relative to the level of machinery that would be expected given a county's cultivated area and the year of observation.

For each machinery category, the logarithm of the number of machines was regressed on the logarithm of total cultivated area and year effects:

\[
\log(machinery_{it} + 1)
=
\alpha
+
\beta \log(cultivated\_area_{it})
+
\gamma_t
+
\varepsilon_{it}
\]

where:

- \(i\) denotes county;
- \(t\) denotes year;
- \(\gamma_t\) represents year fixed effects;
- \(\varepsilon_{it}\) represents the residual component.

The residuals capture whether a county has more or fewer machines than would be expected given its cultivated area and the common conditions of that year.

Residuals were calculated separately for tractors, ploughs, mechanical seeders, and combine harvesters. They were then standardized and averaged to construct the **residual mechanization index**.

A positive value indicates that a county has a higher level of mechanization than expected given its cultivated area and year, while a negative value indicates a lower-than-expected level of mechanization.









### 1. Data Loading and Initial Inspection

In [34]:
import numpy as np
import pandas as pd

In [35]:
df = pd.read_csv("../../data/index/mechanization_residual_index_input.csv")
df.head(2)

,an,judet,emig_masc_nr,emig_fem_nr,emigranti_total_nr,emig_15_64_nr,imig_masc_nr,imig_fem_nr,imigranti_total_nr,imigranti_15_64_nr,...,z_pluguri_1000ha,z_semanatori_1000ha,z_combine_1000ha,indice_mecanizare_vechi,indice_mecanizare_nou,temperatura_medie_anuala_C,schimbare_temp_anuala_C,precipitatii_anuale_mm,schimbare_precipitatii_anuala_mm,schimbare_precipitatii_anuala_pct
0,2012,Alba,143,144,287,222,43,33,76,66,...,0.004986,0.728610,0.706037,0.325602,0.544519,8.933329,0.633381,763.673799,64.982254,9.300564
1,2013,Alba,104,158,262,224,53,41,94,83,...,0.111669,0.764576,0.823547,0.415279,0.548323,8.863476,-0.069853,960.123806,196.450007,25.724335


In [36]:
df.columns

Index(['an', 'judet', 'emig_masc_nr', 'emig_fem_nr', 'emigranti_total_nr',
       'emig_15_64_nr', 'imig_masc_nr', 'imig_fem_nr', 'imigranti_total_nr',
       'imigranti_15_64_nr', 'populatie_15_64', 'populatie_rurala',
       'populatie_totala', 'pondere_rurala', 'rata_emig_def', 'rata_imig_def',
       'rata_emig_15_64', 'rata_imig_15_64', 'intensitate_migratie',
       'sup_grau_ha', 'sup_orz_orzoaica_ha', 'sup_porumb_boabe_ha',
       'sup_floarea_soarelui_ha', 'sup_rapita_ha', 'sup_soia_boabe_ha',
       'sup_totala_cultivata_ha', 'pondere_grau', 'pondere_orz_orzoaica',
       'pondere_porumb_boabe', 'pondere_floarea_soarelui', 'pondere_rapita',
       'pondere_soia_boabe', 'prod_grau_tone', 'prod_orz_orzoaica_tone',
       'prod_porumb_boabe_tone', 'prod_floarea_soarelui_tone',
       'prod_rapita_tone', 'prod_soia_boabe_tone', 'yield_grau',
       'yield_orz_orzoaica', 'yield_porumb_boabe', 'yield_floarea_soarelui',
       'yield_rapita', 'yield_soia_boabe', 'ocupati_total_nr',


In [37]:
df = df.drop(columns=["indice_mecanizare_nou"])

In [38]:
df.columns

Index(['an', 'judet', 'emig_masc_nr', 'emig_fem_nr', 'emigranti_total_nr',
       'emig_15_64_nr', 'imig_masc_nr', 'imig_fem_nr', 'imigranti_total_nr',
       'imigranti_15_64_nr', 'populatie_15_64', 'populatie_rurala',
       'populatie_totala', 'pondere_rurala', 'rata_emig_def', 'rata_imig_def',
       'rata_emig_15_64', 'rata_imig_15_64', 'intensitate_migratie',
       'sup_grau_ha', 'sup_orz_orzoaica_ha', 'sup_porumb_boabe_ha',
       'sup_floarea_soarelui_ha', 'sup_rapita_ha', 'sup_soia_boabe_ha',
       'sup_totala_cultivata_ha', 'pondere_grau', 'pondere_orz_orzoaica',
       'pondere_porumb_boabe', 'pondere_floarea_soarelui', 'pondere_rapita',
       'pondere_soia_boabe', 'prod_grau_tone', 'prod_orz_orzoaica_tone',
       'prod_porumb_boabe_tone', 'prod_floarea_soarelui_tone',
       'prod_rapita_tone', 'prod_soia_boabe_tone', 'yield_grau',
       'yield_orz_orzoaica', 'yield_porumb_boabe', 'yield_floarea_soarelui',
       'yield_rapita', 'yield_soia_boabe', 'ocupati_total_nr',


### 2. Residualization of Machinery Indicators

For each machinery category, the number of machines is adjusted for cultivated area and year effects. The resulting residuals capture whether a county has more or fewer machines than expected given its cultivated area and the year of observation.

In [39]:
import statsmodels.formula.api as smf

In [40]:
df = df.copy()
area_col = "sup_totala_cultivata_ha"

utilaje = {
    "tractoare": "tractoare_nr",
    "pluguri": "pluguri_nr",
    "semanatori": "semanatori_mecanice_nr",
    "combine": "combine_cereale_nr"
}

df["log_suprafata"] = np.log(df[area_col])

for nume, col in utilaje.items():
    df[f"log_{nume}"] = np.log1p(df[col])
    
    model = smf.ols(
        formula=f"log_{nume} ~ log_suprafata + C(an)",
        data=df
    ).fit()
    
    df[f"rezid_{nume}"] = model.resid
    df[f"z_rezid_{nume}"] = (
        df[f"rezid_{nume}"] - df[f"rezid_{nume}"].mean()
    ) / df[f"rezid_{nume}"].std(ddof=1)

df["indice_mecanizare_rezidual"] = df[
    ["z_rezid_tractoare", "z_rezid_pluguri", "z_rezid_semanatori", "z_rezid_combine"]
].mean(axis=1, skipna=False)

In [9]:
df.columns

Index(['an', 'judet', 'emig_masc_nr', 'emig_fem_nr', 'emigranti_total_nr',
       'emig_15_64_nr', 'imig_masc_nr', 'imig_fem_nr', 'imigranti_total_nr',
       'imigranti_15_64_nr', 'populatie_15_64', 'populatie_rurala',
       'populatie_totala', 'pondere_rurala', 'rata_emig_def', 'rata_imig_def',
       'rata_emig_15_64', 'rata_imig_15_64', 'intensitate_migratie',
       'sup_grau_ha', 'sup_orz_orzoaica_ha', 'sup_porumb_boabe_ha',
       'sup_floarea_soarelui_ha', 'sup_rapita_ha', 'sup_soia_boabe_ha',
       'sup_totala_cultivata_ha', 'pondere_grau', 'pondere_orz_orzoaica',
       'pondere_porumb_boabe', 'pondere_floarea_soarelui', 'pondere_rapita',
       'pondere_soia_boabe', 'prod_grau_tone', 'prod_orz_orzoaica_tone',
       'prod_porumb_boabe_tone', 'prod_floarea_soarelui_tone',
       'prod_rapita_tone', 'prod_soia_boabe_tone', 'yield_grau',
       'yield_orz_orzoaica', 'yield_porumb_boabe', 'yield_floarea_soarelui',
       'yield_rapita', 'yield_soia_boabe', 'ocupati_total_nr',




### 3. Comparison with the Density-Based Mechanization Index

The residual mechanization index is compared with the initial density-based index, which measures machinery stocks relative to total cultivated area.

In [41]:
df[["indice_mecanizare_vechi", "indice_mecanizare_rezidual"]].corr()

,indice_mecanizare_vechi,indice_mecanizare_rezidual
indice_mecanizare_vechi,1.000000,0.403942
indice_mecanizare_rezidual,0.403942,1.000000


### Counties with the Highest and Lowest Residual Mechanization Index Values

In [11]:
df.groupby("judet")[[
    "indice_mecanizare_vechi",
    "indice_mecanizare_rezidual"
]].mean().sort_values("indice_mecanizare_rezidual", ascending=False).head(10)

,indice_mecanizare_vechi,indice_mecanizare_rezidual
judet,,
Bihor,-0.029794,1.797615
Timis,-0.536888,1.166924
Mures,0.249953,1.157277
Dolj,-0.572379,1.111236
Olt,-0.465848,1.091818
Arad,-0.403079,0.948738
Satu Mare,-0.131851,0.905391
Maramures,2.921165,0.874954
Teleorman,-0.622289,0.782440


In [12]:
df.groupby("judet")[[
    "indice_mecanizare_vechi",
    "indice_mecanizare_rezidual"
]].mean().sort_values("indice_mecanizare_rezidual", ascending=True).head(10)

,indice_mecanizare_vechi,indice_mecanizare_rezidual
judet,,
Ilfov,-0.411663,-1.885661
Prahova,-0.746506,-1.731969
Tulcea,-0.835761,-1.271039
Buzau,-0.873485,-1.164547
Braila,-0.881755,-1.162342
Bistrita-Nasaud,0.378123,-1.144221
Galati,-0.848558,-1.076469
Ialomita,-0.857104,-0.805016
Neamt,-0.452549,-0.736911


In [13]:
df["schimbare_pondere_ocupati_agri"] = (
    df.groupby("judet")["pondere_ocupati_agri"].diff()
)


In [14]:
df.columns

Index(['an', 'judet', 'emig_masc_nr', 'emig_fem_nr', 'emigranti_total_nr',
       'emig_15_64_nr', 'imig_masc_nr', 'imig_fem_nr', 'imigranti_total_nr',
       'imigranti_15_64_nr', 'populatie_15_64', 'populatie_rurala',
       'populatie_totala', 'pondere_rurala', 'rata_emig_def', 'rata_imig_def',
       'rata_emig_15_64', 'rata_imig_15_64', 'intensitate_migratie',
       'sup_grau_ha', 'sup_orz_orzoaica_ha', 'sup_porumb_boabe_ha',
       'sup_floarea_soarelui_ha', 'sup_rapita_ha', 'sup_soia_boabe_ha',
       'sup_totala_cultivata_ha', 'pondere_grau', 'pondere_orz_orzoaica',
       'pondere_porumb_boabe', 'pondere_floarea_soarelui', 'pondere_rapita',
       'pondere_soia_boabe', 'prod_grau_tone', 'prod_orz_orzoaica_tone',
       'prod_porumb_boabe_tone', 'prod_floarea_soarelui_tone',
       'prod_rapita_tone', 'prod_soia_boabe_tone', 'yield_grau',
       'yield_orz_orzoaica', 'yield_porumb_boabe', 'yield_floarea_soarelui',
       'yield_rapita', 'yield_soia_boabe', 'ocupati_total_nr',


### Intermediate Dataset with Both Mechanization Indices

During the original analysis, an intermediate dataset was exported at this stage containing both the density-based and residual mechanization indices.

For the reorganized GitHub version of the project, this intermediate export step is omitted to avoid generating redundant files. The corresponding dataset is provided directly in the repository and is used in the subsequent analysis.

### 4. Internal Consistency of the Residual Mechanization Index

Cronbach's alpha is used to assess the internal consistency of the four standardized residual machinery indicators included in the residual mechanization index.

In [44]:
def cronbach_alpha(data):
    data = data.astype(float)
    n_items = data.shape[1]
    
    item_variances = data.var(axis=0, ddof=1)
    total_score = data.sum(axis=1)
    total_variance = total_score.var(ddof=1)
    
    alpha = (n_items / (n_items - 1)) * (
        1 - item_variances.sum() / total_variance
    )
    
    return alpha

In [45]:
items_rezid = [
    "z_rezid_tractoare",
    "z_rezid_pluguri",
    "z_rezid_semanatori",
    "z_rezid_combine"
]

df_alpha_z = df[items_rezid].dropna()

alpha = cronbach_alpha(df_alpha_z)

print("Cronbach's alpha for the residual mechanization index:", round(alpha, 3))

Cronbach's alpha for the residual mechanization index: 0.929


In [46]:
import pandas as pd
import matplotlib.pyplot as plt

items_resid = [
    "z_rezid_tractoare",
    "z_rezid_pluguri",
    "z_rezid_semanatori",
    "z_rezid_combine"
]

english_names_resid = {
    "z_rezid_tractoare": "Residual tractors",
    "z_rezid_pluguri": "Residual ploughs",
    "z_rezid_semanatori": "Residual mechanical seeders",
    "z_rezid_combine": "Residual combine harvesters"
}


df_alpha_z = df[items_resid].dropna()

# overall Alpha
overall_alpha = cronbach_alpha(df_alpha_z)

# Alpha if item deleted
rows = []

rows.append({
    "Indicator": "All residual indicators",
    "Cronbach's alpha": round(overall_alpha, 3)
})

for item in items_resid:
    reduced_items = [x for x in items_resid if x != item]
    alpha_reduced = cronbach_alpha(df_alpha_z[reduced_items])
    
    rows.append({
        "Indicator": f"Without {english_names_resid[item]}",
        "Cronbach's alpha": round(alpha_reduced, 3)
    })

cronbach_resid_table = pd.DataFrame(rows)

display(cronbach_resid_table)



,Indicator,Cronbach's alpha
0,All residual indicators,0.929
1,Without Residual tractors,0.900
2,Without Residual ploughs,0.883
3,Without Residual mechanical seeders,0.912
4,Without Residual combine harvesters,0.932
